# Music Artist Recommendation System

A hybrid recommender system built on the [Last.fm Dataset (HetRec 2011)](https://grouplens.org/datasets/hetrec-2011/) using:

- **Item-based Collaborative Filtering** — recommends artists similar to ones you already play
- **User-based Collaborative Filtering** — recommends artists liked by users similar to you
- **Content-based Filtering** — recommends artists sharing the same genre/mood tags

Similarity is computed with **cosine similarity** across all three approaches.

### Dataset files required (place in `data/` folder)
```
data/
  artists.dat
  user_artists.dat
  tags.dat
  user_taggedartists.dat
  user_taggedartists-timestamps.dat
  user_friends.dat
```
Download from: https://grouplens.org/datasets/hetrec-2011/

In [13]:
# Install dependencies (uncomment if running for the first time)
# !pip install pandas numpy scikit-learn matplotlib seaborn

In [14]:
import os
import pandas as pd
import numpy as np

# ── Dataset path ──────────────────────────────────────────────────────────────
# All .dat files should be placed in a 'data/' subfolder next to this notebook.
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

def data_path(filename):
    return os.path.join(DATA_DIR, filename)

# ── Load and explore user_artists.dat ────────────────────────────────────────
user_artists_df = pd.read_csv(data_path('user_artists.dat'), sep='\t')
print("--- user_artists_df Info ---")
user_artists_df.info()
print("\n--- user_artists_df Head ---")
print(user_artists_df.head())
print(f"\nUnique users: {user_artists_df['userID'].nunique()}")
print(f"Unique artists: {user_artists_df['artistID'].nunique()}")
print("\nMissing values in user_artists_df:")
print(user_artists_df.isnull().sum())

# ── Load and explore artists.dat ─────────────────────────────────────────────
artists_df = pd.read_csv(data_path('artists.dat'), sep='\t')
print("\n\n--- artists_df Info ---")
artists_df.info()
print("\n--- artists_df Head ---")
print(artists_df.head())
print(f"\nUnique artists in artists_df: {artists_df['id'].nunique()}")
print("\nMissing values in artists_df:")
print(artists_df.isnull().sum())

# ── Load and explore tags.dat ─────────────────────────────────────────────────
tags_df = pd.read_csv(data_path('tags.dat'), sep='\t', encoding='latin1')
print("\n\n--- tags_df Info ---")
tags_df.info()
print("\n--- tags_df Head ---")
print(tags_df.head())
print(f"\nUnique tags: {tags_df['tagID'].nunique()}")
print("\nMissing values in tags_df:")
print(tags_df.isnull().sum())

# ── Load and explore user_taggedartists.dat ───────────────────────────────────
user_taggedartists_df = pd.read_csv(data_path('user_taggedartists.dat'), sep='\t', encoding='latin1')
print("\n\n--- user_taggedartists_df Info ---")
user_taggedartists_df.info()
print("\n--- user_taggedartists_df Head ---")
print(user_taggedartists_df.head())
print(f"\nUnique users in user_taggedartists_df: {user_taggedartists_df['userID'].nunique()}")
print(f"Unique artists in user_taggedartists_df: {user_taggedartists_df['artistID'].nunique()}")
print(f"Unique tags in user_taggedartists_df: {user_taggedartists_df['tagID'].nunique()}")
print("\nMissing values in user_taggedartists_df:")
print(user_taggedartists_df.isnull().sum())

# ── Load and explore user_friends.dat ────────────────────────────────────────
user_friends_df = pd.read_csv(data_path('user_friends.dat'), sep='\t', encoding='latin1')
print("\n\n--- user_friends_df Info ---")
user_friends_df.info()
print("\n--- user_friends_df Head ---")
print(user_friends_df.head())
print(f"\nUnique users in user_friends_df: {user_friends_df['userID'].nunique()}")
print(f"Unique friends in user_friends_df: {user_friends_df['friendID'].nunique()}")
print("\nMissing values in user_friends_df:")
print(user_friends_df.isnull().sum())

# ── Load and explore user_taggedartists-timestamps.dat ───────────────────────
user_taggedartists_timestamps_df = pd.read_csv(data_path('user_taggedartists-timestamps.dat'), sep='\t', encoding='latin1')
print("\n\n--- user_taggedartists-timestamps_df Info ---")
user_taggedartists_timestamps_df.info()
print("\n--- user_taggedartists-timestamps_df Head ---")
print(user_taggedartists_timestamps_df.head())
print("\nMissing values in user_taggedartists-timestamps_df:")
print(user_taggedartists_timestamps_df.isnull().sum())

--- user_artists_df Info ---
<class 'pandas.DataFrame'>
RangeIndex: 92834 entries, 0 to 92833
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   userID    92834 non-null  int64
 1   artistID  92834 non-null  int64
 2   weight    92834 non-null  int64
dtypes: int64(3)
memory usage: 2.1 MB

--- user_artists_df Head ---
   userID  artistID  weight
0       2        51   13883
1       2        52   11690
2       2        53   11351
3       2        54   10300
4       2        55    8983

Unique users: 1892
Unique artists: 17632

Missing values in user_artists_df:
userID      0
artistID    0
weight      0
dtype: int64


--- artists_df Info ---
<class 'pandas.DataFrame'>
RangeIndex: 17632 entries, 0 to 17631
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          17632 non-null  int64
 1   name        17632 non-null  str  
 2   url         17632 non-null  str  
 3

In [15]:
# ── 1. Rename 'id' column in artists_df to 'artistID' for consistent merging ─
artists_df.rename(columns={'id': 'artistID'}, inplace=True)
print("--- artists_df after renaming 'id' to 'artistID' ---")
print(artists_df.head())

# ── 2. Merge user_artists_df with artists_df to add artist names ──────────────
df_interactions = pd.merge(user_artists_df, artists_df[['artistID', 'name']],
                           on='artistID', how='left')
print("\n--- df_interactions (merged) Head ---")
print(df_interactions.head())
print("\n--- df_interactions Info ---")
df_interactions.info()
print("\nMissing artist names after merge (should be 0 or very low):", df_interactions['name'].isnull().sum())

# ── 3. Consolidate duplicate user-artist interactions by summing play counts ──
# A user may appear multiple times for the same artist; we sum the 'weight' column.
duplicate_interactions_count = df_interactions.duplicated(subset=['userID', 'artistID']).sum()
print(f"\nDuplicate user-artist interactions before consolidation: {duplicate_interactions_count}")

if duplicate_interactions_count > 0:
    print("Consolidating duplicates by summing 'weight'...")
    df_interactions_clean = df_interactions.groupby(['userID', 'artistID', 'name'], as_index=False)['weight'].sum()
    print(f"Unique user-artist interactions after consolidation: {len(df_interactions_clean)}")
else:
    print("No duplicates found.")
    df_interactions_clean = df_interactions.copy()
    # Keep only artists with at least 20 users — reduces matrix size significantly
min_users = 20
popular_artists = user_artists_df.groupby('artistID')['userID'].nunique()
popular_artist_ids = popular_artists[popular_artists >= min_users].index
df_interactions_clean = df_interactions_clean[df_interactions_clean['artistID'].isin(popular_artist_ids)]

--- artists_df after renaming 'id' to 'artistID' ---
   artistID               name                                         url  \
0         1       MALICE MIZER       http://www.last.fm/music/MALICE+MIZER   
1         2    Diary of Dreams    http://www.last.fm/music/Diary+of+Dreams   
2         3  Carpathian Forest  http://www.last.fm/music/Carpathian+Forest   
3         4       Moi dix Mois       http://www.last.fm/music/Moi+dix+Mois   
4         5        Bella Morte        http://www.last.fm/music/Bella+Morte   

                                          pictureURL  
0    http://userserve-ak.last.fm/serve/252/10808.jpg  
1  http://userserve-ak.last.fm/serve/252/3052066.jpg  
2  http://userserve-ak.last.fm/serve/252/40222717...  
3  http://userserve-ak.last.fm/serve/252/54697835...  
4  http://userserve-ak.last.fm/serve/252/14789013...  

--- df_interactions (merged) Head ---
   userID  artistID  weight           name
0       2        51   13883    Duran Duran
1       2        52   1

In [16]:
# ── Enrich tag data with artist names ────────────────────────────────────────
df_user_artist_tags = pd.merge(user_taggedartists_df, tags_df, on='tagID', how='left')
print("--- df_user_artist_tags (merged with tags) Head ---")
print(df_user_artist_tags.head())
print("\nMissing tag values after merge:", df_user_artist_tags['tagValue'].isnull().sum())

df_user_artist_tags_with_names = pd.merge(df_user_artist_tags, artists_df[['artistID', 'name']],
                                          on='artistID', how='left')
print("\n--- df_user_artist_tags_with_names Head ---")
print(df_user_artist_tags_with_names.head())
print("\nMissing artist names in tagged data:", df_user_artist_tags_with_names['name'].isnull().sum())

# ── Parse timestamps (milliseconds → datetime) ────────────────────────────────
user_taggedartists_timestamps_df['datetime'] = pd.to_datetime(
    user_taggedartists_timestamps_df['timestamp'], unit='ms'
)
print("\n--- user_taggedartists-timestamps_df with datetime column ---")
print(user_taggedartists_timestamps_df.head())

--- df_user_artist_tags (merged with tags) Head ---
   userID  artistID  tagID  day  month  year          tagValue
0       2        52     13    1      4  2009          chillout
1       2        52     15    1      4  2009         downtempo
2       2        52     18    1      4  2009        electronic
3       2        52     21    1      4  2009          trip-hop
4       2        52     41    1      4  2009  female vovalists

Missing tag values after merge: 0

--- df_user_artist_tags_with_names Head ---
   userID  artistID  tagID  day  month  year          tagValue       name
0       2        52     13    1      4  2009          chillout  Morcheeba
1       2        52     15    1      4  2009         downtempo  Morcheeba
2       2        52     18    1      4  2009        electronic  Morcheeba
3       2        52     21    1      4  2009          trip-hop  Morcheeba
4       2        52     41    1      4  2009  female vovalists  Morcheeba

Missing artist names in tagged data: 1538

--

In [17]:
# ── Clean friend relationships (remove exact duplicates) ─────────────────────
print("--- user_friends_df Head ---")
print(user_friends_df.head())
user_friends_df.info()
print("\nMissing values:", user_friends_df.isnull().sum().to_dict())

duplicate_friends = user_friends_df.duplicated().sum()
print(f"\nDuplicate friend relationships: {duplicate_friends}")

if duplicate_friends > 0:
    user_friends_df_clean = user_friends_df.drop_duplicates().reset_index(drop=True)
    print(f"Friend relationships after deduplication: {len(user_friends_df_clean)}")
else:
    user_friends_df_clean = user_friends_df.copy()

--- user_friends_df Head ---
   userID  friendID
0       2       275
1       2       428
2       2       515
3       2       761
4       2       831
<class 'pandas.DataFrame'>
RangeIndex: 25434 entries, 0 to 25433
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   userID    25434 non-null  int64
 1   friendID  25434 non-null  int64
dtypes: int64(2)
memory usage: 397.5 KB

Missing values: {'userID': 0, 'friendID': 0}

Duplicate friend relationships: 0


In [18]:
from sklearn.model_selection import train_test_split

# Features: userID and artistID; target: play-count weight
X = df_interactions_clean[['userID', 'artistID']]
y = df_interactions_clean['weight']

# 75% train / 25% test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("--- Data Split Summary ---")
print(f"Total interactions : {len(df_interactions_clean)}")
print(f"Training (75%)     : {len(X_train)}")
print(f"Testing  (25%)     : {len(X_test)}")

--- Data Split Summary ---
Total interactions : 53234
Training (75%)     : 39925
Testing  (25%)     : 13309


## Item-based Collaborative Filtering

Build a user-item matrix and compute **cosine similarity between artists**.  
To recommend for a user, we score each unseen artist by a weighted sum of similarities to the user's already-played artists, using play count as the weight.

In [19]:
import numpy as np
import pandas as pd

# Build user-item matrix: rows = users, columns = artists, values = play count
# Missing entries (user never played that artist) are filled with 0.
user_item_matrix = df_interactions_clean.pivot_table(
    index='userID', columns='artistID', values='weight'
).fillna(0)

print("--- User-Item Matrix (first 5×5) ---")
print(user_item_matrix.iloc[:5, :5])
print(f"\nShape: {user_item_matrix.shape}  (users × artists)")

artist_ids = user_item_matrix.columns

--- User-Item Matrix (first 5×5) ---
artistID   7    9    15   25   30
userID                           
2         0.0  0.0  0.0  0.0  0.0
3         0.0  0.0  0.0  0.0  0.0
4         0.0  0.0  0.0  0.0  0.0
5         0.0  0.0  0.0  0.0  0.0
6         0.0  0.0  0.0  0.0  0.0

Shape: (1869, 804)  (users × artists)


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

# Transpose so artists become rows, then compute pairwise cosine similarity
item_similarity_matrix = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity_matrix,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

print("--- Item-Item Cosine Similarity Matrix (first 5×5) ---")
print(item_similarity_df.iloc[:5, :5])
print(f"\nShape: {item_similarity_df.shape}")

--- Item-Item Cosine Similarity Matrix (first 5×5) ---
artistID        7         9         15        25        30
artistID                                                  
7         1.000000  0.059799  0.022772  0.035709  0.007243
9         0.059799  1.000000  0.006761  0.047205  0.069225
15        0.022772  0.006761  1.000000  0.363176  0.003695
25        0.035709  0.047205  0.363176  1.000000  0.001652
30        0.007243  0.069225  0.003695  0.001652  1.000000

Shape: (804, 804)


In [21]:
def get_item_based_recommendations(user_id, num_recommendations=10):
    """Return top-N artist recommendations for a user using item-based CF."""
    user_listened_artists = df_interactions_clean[df_interactions_clean['userID'] == user_id]

    if user_listened_artists.empty:
        print(f"User {user_id} is a cold-start user (no listening history).")
        return []

    user_artist_weights = dict(zip(user_listened_artists['artistID'], user_listened_artists['weight']))
    recommendation_scores = {}

    for listened_artist_id, listened_weight in user_artist_weights.items():
        if listened_artist_id not in item_similarity_df.columns:
            continue

        similar_artists_scores = item_similarity_df[listened_artist_id]

        for similar_artist_id, similarity_score in similar_artists_scores.items():
            if similar_artist_id == listened_artist_id:
                continue
            if similar_artist_id in user_artist_weights:
                continue

            score = similarity_score * listened_weight
            recommendation_scores[similar_artist_id] = recommendation_scores.get(similar_artist_id, 0) + score

    top_recommendations = sorted(
        [(score, aid) for aid, score in recommendation_scores.items() if score > 0],
        reverse=True
    )[:num_recommendations]

    artist_id_to_name = dict(zip(artists_df['artistID'], artists_df['name']))
    return [(artist_id_to_name.get(aid, f"Artist ID {aid}"), score) for score, aid in top_recommendations]


# --- Sample output ---
# Users 2, 5, 170 have listening history; 3000 is a cold-start example.
print("--- Item-Based Recommendations ---")
for user_id in [2, 5, 170, 3000]:
    print(f"\nRecommendations for User {user_id}:")
    recs = get_item_based_recommendations(user_id, num_recommendations=5)
    if recs:
        for artist_name, score in recs:
            print(f"  - {artist_name} (Score: {score:.2f})")
    else:
        print("  No recommendations available.")

--- Item-Based Recommendations ---

Recommendations for User 2:
  - Eurythmics (Score: 17134.58)
  - The Human League (Score: 16702.99)
  - The Cure (Score: 16465.26)
  - Roxy Music (Score: 15719.56)
  - Pet Shop Boys (Score: 15457.23)

Recommendations for User 5:
  - Sufjan Stevens (Score: 1293.57)
  - Pixies (Score: 1143.75)
  - The Shins (Score: 1138.71)
  - The Strokes (Score: 1120.02)
  - Animal Collective (Score: 1091.76)

Recommendations for User 170:
  - Kreator (Score: 4671.29)
  - Ozzy Osbourne (Score: 4357.19)
  - A Perfect Circle (Score: 3966.05)
  - Dark Tranquillity (Score: 3697.22)
  - Soundgarden (Score: 3643.13)

Recommendations for User 3000:
User 3000 is a cold-start user (no listening history).
  No recommendations available.


## User-based Collaborative Filtering

Compute **cosine similarity between users** and recommend artists that highly similar users have played but the target user has not.

In [22]:
# Convert to float32 to halve memory usage before computing similarity
user_item_matrix_f32 = user_item_matrix.astype('float32')

user_similarity_matrix = cosine_similarity(user_item_matrix_f32)
user_similarity_df = pd.DataFrame(
    user_similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("--- User-User Cosine Similarity Matrix (first 5×5) ---")
print(user_similarity_df.iloc[:5, :5])
print(f"\nShape: {user_similarity_df.shape}")

--- User-User Cosine Similarity Matrix (first 5×5) ---
userID         2    3         4         5         6
userID                                             
2       1.000000  0.0  0.181693  0.037154  0.013891
3       0.000000  1.0  0.000000  0.000000  0.000000
4       0.181693  0.0  1.000000  0.098822  0.012298
5       0.037154  0.0  0.098822  1.000000  0.000000
6       0.013891  0.0  0.012298  0.000000  1.000000

Shape: (1869, 1869)


In [23]:
def get_user_based_recommendations(user_id, num_recommendations=10, k_neighbors=50):
    """Return top-N artist recommendations for a user using user-based CF."""
    user_listened_artists_ids = set(
        df_interactions_clean[df_interactions_clean['userID'] == user_id]['artistID']
    )

    if user_id not in user_similarity_df.index:
        print(f"User {user_id} not in similarity matrix (cold-start).")
        return []

    user_similarities = user_similarity_df[user_id].drop(user_id)
    top_k_similar_users = user_similarities[user_similarities > 0].nlargest(k_neighbors).index.tolist()

    if not top_k_similar_users:
        print(f"No similar users found for User {user_id}.")
        return []

    recommendation_scores = {}
    artist_id_to_name = dict(zip(artists_df['artistID'], artists_df['name']))

    for similar_user_id in top_k_similar_users:
        similarity_score = user_similarity_df.loc[user_id, similar_user_id]
        similar_user_artists = df_interactions_clean[df_interactions_clean['userID'] == similar_user_id]

        for _, row in similar_user_artists.iterrows():
            artist_id = row['artistID']
            if artist_id in user_listened_artists_ids:
                continue
            score = similarity_score * row['weight']
            recommendation_scores[artist_id] = recommendation_scores.get(artist_id, 0) + score

    top_recommendations = sorted(
        [(score, aid) for aid, score in recommendation_scores.items() if score > 0],
        reverse=True
    )[:num_recommendations]

    return [(artist_id_to_name.get(aid, f"Artist ID {aid}"), score) for score, aid in top_recommendations]


# --- Sample output ---
print("--- User-Based Recommendations ---")
for user_id in [2, 5, 170]:
    print(f"\nRecommendations for User {user_id}:")
    recs = get_user_based_recommendations(user_id, num_recommendations=5, k_neighbors=50)
    if recs:
        for artist_name, score in recs:
            print(f"  - {artist_name} (Score: {score:.2f})")
    else:
        print("  No recommendations available.")

--- User-Based Recommendations ---

Recommendations for User 2:
  - U2 (Score: 14762.87)
  - Pet Shop Boys (Score: 12202.09)
  - The Cure (Score: 11851.33)
  - Britney Spears (Score: 11577.50)
  - Christina Aguilera (Score: 9859.19)

Recommendations for User 5:
  - Coldplay (Score: 16145.50)
  - Sigur Rós (Score: 7367.96)
  - The Strokes (Score: 5164.23)
  - Elliott Smith (Score: 5075.84)
  - Animal Collective (Score: 5075.15)

Recommendations for User 170:
  - Opeth (Score: 6455.55)
  - Judas Priest (Score: 5133.62)
  - Led Zeppelin (Score: 5023.60)
  - System of a Down (Score: 4961.05)
  - Pink Floyd (Score: 4945.93)


## Evaluation — Predicting Play Counts

We use **Mean Absolute Error (MAE)** and **Root Mean Squared Error (RMSE)** on the held-out test set to evaluate prediction accuracy.

In [24]:
def predict_item_based(user_id, artist_id):
    """Predict play-count weight for a user-artist pair using item-based CF."""
    if artist_id not in item_similarity_df.columns or user_id not in user_item_matrix.index:
        return 0

    user_listened_artists = df_interactions_clean[df_interactions_clean['userID'] == user_id]
    if user_listened_artists.empty:
        return 0

    user_artist_weights = dict(zip(user_listened_artists['artistID'], user_listened_artists['weight']))
    target_artist_similarities = item_similarity_df[artist_id]

    numerator = 0
    denominator = 0

    for listened_artist_id, listened_weight in user_artist_weights.items():
        if listened_artist_id not in target_artist_similarities.index:
            continue
        sim = target_artist_similarities[listened_artist_id]
        if sim > 0:
            numerator   += sim * listened_weight
            denominator += sim

    return numerator / denominator if denominator > 0 else 0


def predict_user_based(user_id, artist_id, k_neighbors=50):
    """Predict play-count weight for a user-artist pair using user-based CF."""
    if user_id not in user_similarity_df.index:
        return 0

    user_similarities = user_similarity_df[user_id].drop(user_id)
    top_k = user_similarities[user_similarities > 0].nlargest(k_neighbors)

    numerator = 0
    denominator = 0

    for neighbor_id, sim in top_k.items():
        neighbor_data = df_interactions_clean[
            (df_interactions_clean['userID'] == neighbor_id) &
            (df_interactions_clean['artistID'] == artist_id)
        ]
        if not neighbor_data.empty:
            weight = neighbor_data.iloc[0]['weight']
            numerator   += sim * weight
            denominator += sim

    return numerator / denominator if denominator > 0 else 0

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

# Sample a subset of the test set for speed (full evaluation is slow on large data)
EVAL_SAMPLE = 500
test_sample = pd.concat([X_test, y_test], axis=1).sample(n=min(EVAL_SAMPLE, len(X_test)), random_state=42)

# Item-based predictions
item_preds = [
    predict_item_based(row['userID'], row['artistID'])
    for _, row in test_sample.iterrows()
]
item_mae  = mean_absolute_error(test_sample['weight'], item_preds)
item_rmse = root_mean_squared_error(test_sample['weight'], item_preds)

# User-based predictions
user_preds = [
    predict_user_based(row['userID'], row['artistID'])
    for _, row in test_sample.iterrows()
]
user_mae  = mean_absolute_error(test_sample['weight'], user_preds)
user_rmse = root_mean_squared_error(test_sample['weight'], user_preds)

print("--- Evaluation Results (sample of", EVAL_SAMPLE, "interactions) ---")
print(f"Item-based CF  |  MAE: {item_mae:.2f}  |  RMSE: {item_rmse:.2f}")
print(f"User-based CF  |  MAE: {user_mae:.2f}  |  RMSE: {user_rmse:.2f}")

--- Evaluation Results (sample of 500 interactions) ---
Item-based CF  |  MAE: 645.55  |  RMSE: 2993.57
User-based CF  |  MAE: 1274.18  |  RMSE: 4794.48


## Explainability

These functions show *why* a particular artist was recommended — either the bridge artists (item-based) or the similar users who drove the recommendation (user-based).

In [27]:
def explain_item_based_recommendation(user_id, recommended_artist_id, num_reasons=3):
    """Explain an item-based recommendation by showing which listened artists drove it."""
    recommended_artist_name = artists_df[artists_df['artistID'] == recommended_artist_id]['name'].iloc[0]
    print(f"\n--- Why was '{recommended_artist_name}' recommended to User {user_id}? ---")

    user_listened = df_interactions_clean[df_interactions_clean['userID'] == user_id]
    if user_listened.empty:
        print("No listening history found.")
        return

    if recommended_artist_id not in item_similarity_df.columns:
        print("Recommended artist not in similarity matrix.")
        return

    similarities_to_recommended = item_similarity_df[recommended_artist_id]
    artist_id_to_name = dict(zip(artists_df['artistID'], artists_df['name']))

    driving_artists = []
    for _, row in user_listened.iterrows():
        listened_artist_id = row['artistID']
        if listened_artist_id in similarities_to_recommended.index:
            sim = similarities_to_recommended[listened_artist_id]
            if sim > 0:
                driving_artists.append((sim, listened_artist_id, row['weight']))

    driving_artists.sort(reverse=True)
    print(f"Top reasons (artists you've played that are similar to '{recommended_artist_name}'):")
    for sim, aid, weight in driving_artists[:num_reasons]:
        name = artist_id_to_name.get(aid, f"Artist {aid}")
        print(f"  - '{name}'  (play count: {weight},  similarity: {sim:.2f})")


# Examples
for artist_name in ['Sufjan Stevens', 'Kreator']:
    row = artists_df[artists_df['name'] == artist_name]
    if not row.empty:
        explain_item_based_recommendation(user_id=5, recommended_artist_id=row['artistID'].iloc[0])


--- Why was 'Sufjan Stevens' recommended to User 5? ---
Top reasons (artists you've played that are similar to 'Sufjan Stevens'):
  - 'The Decemberists'  (play count: 222,  similarity: 0.52)
  - 'Death Cab for Cutie'  (play count: 185,  similarity: 0.45)
  - 'Radiohead'  (play count: 884,  similarity: 0.33)

--- Why was 'Kreator' recommended to User 5? ---
Top reasons (artists you've played that are similar to 'Kreator'):
  - 'Red Hot Chili Peppers'  (play count: 170,  similarity: 0.37)
  - 'Muse'  (play count: 826,  similarity: 0.02)
  - 'Radiohead'  (play count: 884,  similarity: 0.02)


In [28]:
def explain_user_based_recommendation(user_id, recommended_artist_id, num_reasons=3, k_neighbors=5):
    """Explain a user-based recommendation by showing which similar users listened to the artist."""
    recommended_artist_name = artists_df[artists_df['artistID'] == recommended_artist_id]['name'].iloc[0]
    print(f"\n--- Why was '{recommended_artist_name}' recommended to User {user_id}? ---")

    if user_id not in user_similarity_df.index:
        print("User not in similarity matrix.")
        return

    user_listened_ids = set(df_interactions_clean[df_interactions_clean['userID'] == user_id]['artistID'])
    user_similarities = user_similarity_df[user_id].drop(user_id)
    top_neighbors = user_similarities[user_similarities > 0].nlargest(k_neighbors).index.tolist()
    artist_id_to_name = dict(zip(df_interactions_clean['artistID'], df_interactions_clean['name']))

    print(f"Recommendation driven by users similar to you who listened to '{recommended_artist_name}':")
    reasons_shown = 0

    for neighbor_id in top_neighbors:
        sim = user_similarity_df.loc[user_id, neighbor_id]
        neighbor_artists = df_interactions_clean[df_interactions_clean['userID'] == neighbor_id]
        neighbor_artist_ids = set(neighbor_artists['artistID'])

        if recommended_artist_id in neighbor_artist_ids:
            common = user_listened_ids.intersection(neighbor_artist_ids)
            common_names = [artist_id_to_name.get(aid, str(aid)) for aid in list(common)[:3]]
            print(f"  - User {neighbor_id} (similarity: {sim:.2f}) — you both enjoy: {', '.join(common_names)}")
            reasons_shown += 1
            if reasons_shown >= num_reasons:
                break


# Examples
for artist_name in ['U2', 'Coldplay', 'Pink Floyd']:
    row = artists_df[artists_df['name'] == artist_name]
    if not row.empty:
        explain_user_based_recommendation(user_id=2, recommended_artist_id=row['artistID'].iloc[0])


--- Why was 'U2' recommended to User 2? ---
Recommendation driven by users similar to you who listened to 'U2':
  - User 1210 (similarity: 0.52) — you both enjoy: Japan, Madonna, Depeche Mode

--- Why was 'Coldplay' recommended to User 2? ---
Recommendation driven by users similar to you who listened to 'Coldplay':
  - User 1866 (similarity: 0.51) — you both enjoy: Lady Gaga, Coldplay, Madonna

--- Why was 'Pink Floyd' recommended to User 2? ---
Recommendation driven by users similar to you who listened to 'Pink Floyd':


## Bonus 1 — Smart Playlist Generator (Item-to-Item)

Given a seed artist, generate a playlist of the most similar artists based on collaborative filtering signals.

In [29]:
def generate_smart_playlist_item_to_item(seed_artist_name, num_songs=7):
    """Generate a playlist of artists similar to the seed artist (item-to-item CF)."""
    print(f"\n--- Playlist based on '{seed_artist_name}' (Item-to-Item CF) ---")

    seed_row = artists_df[artists_df['name'].str.lower() == seed_artist_name.lower()]
    if seed_row.empty:
        print(f"Artist '{seed_artist_name}' not found.")
        return []

    seed_id = seed_row['artistID'].iloc[0]
    if seed_id not in item_similarity_df.columns:
        print(f"'{seed_artist_name}' not in similarity matrix (too few interactions?).")
        return []

    scores = item_similarity_df[seed_id].drop(seed_id)
    top = scores[scores > 0].nlargest(num_songs)

    artist_id_to_name = dict(zip(artists_df['artistID'], artists_df['name']))
    return [f"{artist_id_to_name.get(aid, aid)} (similarity: {s:.2f})" for aid, s in top.items()]


for seed in ['Duran Duran', 'Radiohead', 'Metallica', 'Beirut', 'Britney Spears']:
    playlist = generate_smart_playlist_item_to_item(seed, num_songs=7)
    if playlist:
        print('\n'.join(f'  {t}' for t in playlist))


--- Playlist based on 'Duran Duran' (Item-to-Item CF) ---
  Spandau Ballet (similarity: 0.85)
  George Michael (similarity: 0.79)
  Roxy Music (similarity: 0.72)
  The Human League (similarity: 0.70)
  Eurythmics (similarity: 0.69)
  The Cure (similarity: 0.67)
  Tears for Fears (similarity: 0.66)

--- Playlist based on 'Radiohead' (Item-to-Item CF) ---
  Thom Yorke (similarity: 0.56)
  Pixies (similarity: 0.44)
  Death Cab for Cutie (similarity: 0.39)
  Interpol (similarity: 0.35)
  Arcade Fire (similarity: 0.35)
  Sufjan Stevens (similarity: 0.33)
  The Decemberists (similarity: 0.33)

--- Playlist based on 'Metallica' (Item-to-Item CF) ---
  Machine Head (similarity: 0.37)
  Ozzy Osbourne (similarity: 0.36)
  Kreator (similarity: 0.33)
  Infected Mushroom (similarity: 0.31)
  Blind Guardian (similarity: 0.29)
  The Offspring (similarity: 0.29)
  Motörhead (similarity: 0.27)

--- Playlist based on 'Beirut' (Item-to-Item CF) ---
  Bon Iver (similarity: 0.43)
  Chico Buarque (similari

## Bonus 2 — Content-Based Filtering (Tag Similarity)

Build an **artist-tag matrix** and compute cosine similarity based on crowd-sourced genre/mood tags.  
This complements CF by finding artists that share musical characteristics — useful when play-count signals are sparse.

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

# Build artist-tag matrix: rows = artists, columns = unique tags, values = tag count
artist_tag_counts = df_user_artist_tags_with_names.groupby(
    ['artistID', 'tagValue']
).size().reset_index(name='tag_count')

artist_tag_matrix = artist_tag_counts.pivot_table(
    index='artistID', columns='tagValue', values='tag_count'
).fillna(0)

print("--- Artist-Tag Matrix (first 5×5) ---")
print(artist_tag_matrix.iloc[:5, :5])
print(f"\nShape: {artist_tag_matrix.shape}  (artists × unique tags)")

# Compute cosine similarity between artists based on their tag profiles
tag_sim_matrix = cosine_similarity(artist_tag_matrix)
tag_based_artist_similarity_df = pd.DataFrame(
    tag_sim_matrix,
    index=artist_tag_matrix.index,
    columns=artist_tag_matrix.index
)

print("\n--- Tag-Based Artist Similarity (first 5×5) ---")
print(tag_based_artist_similarity_df.iloc[:5, :5])

--- Artist-Tag Matrix (first 5×5) ---
tagValue  '80s  -pearl fashion music  0 play yet   00  00's
artistID                                                   
1          0.0                   0.0         0.0  0.0   0.0
2          0.0                   0.0         0.0  0.0   0.0
3          0.0                   0.0         0.0  0.0   0.0
4          0.0                   0.0         0.0  0.0   0.0
5          0.0                   0.0         0.0  0.0   0.0

Shape: (12523, 9749)  (artists × unique tags)

--- Tag-Based Artist Similarity (first 5×5) ---
artistID         1         2    3         4         5
artistID                                             
1         1.000000  0.133333  0.0  0.894586  0.082690
2         0.133333  1.000000  0.0  0.151911  0.744208
3         0.000000  0.000000  1.0  0.000000  0.000000
4         0.894586  0.151911  0.0  1.000000  0.094211
5         0.082690  0.744208  0.0  0.094211  1.000000


In [31]:
def generate_tag_based_playlist(seed_artist_name, num_songs=7):
    """Generate a playlist of artists sharing similar tags to the seed artist."""
    print(f"\n--- Tag-Based Playlist: '{seed_artist_name}' ---")

    seed_row = artists_df[artists_df['name'].str.lower() == seed_artist_name.lower()]
    if seed_row.empty:
        print(f"Artist '{seed_artist_name}' not found.")
        return []

    seed_id = seed_row['artistID'].iloc[0]
    if seed_id not in tag_based_artist_similarity_df.columns:
        print(f"'{seed_artist_name}' not in tag similarity matrix (no tags?).")
        return []

    scores = tag_based_artist_similarity_df[seed_id].drop(seed_id)
    top = scores[scores > 0].nlargest(num_songs)

    artist_id_to_name = dict(zip(artists_df['artistID'], artists_df['name']))
    return [f"{artist_id_to_name.get(aid, aid)} (tag similarity: {s:.2f})" for aid, s in top.items()]


for seed in ['Metallica', 'Radiohead', 'Britney Spears', 'Beirut']:
    playlist = generate_tag_based_playlist(seed, num_songs=5)
    if playlist:
        print('\n'.join(f'  {t}' for t in playlist))


--- Tag-Based Playlist: 'Metallica' ---
  Megadeth (tag similarity: 0.86)
  Коррозия Металла (tag similarity: 0.84)
  Anthrax (tag similarity: 0.84)
  Hellyeah (tag similarity: 0.83)
  Armored Saint (tag similarity: 0.82)

--- Tag-Based Playlist: 'Radiohead' ---
  Placebo (tag similarity: 0.91)
  Muse (tag similarity: 0.89)
  Coldplay (tag similarity: 0.88)
  Weezer (tag similarity: 0.84)
  Beck (tag similarity: 0.83)

--- Tag-Based Playlist: 'Britney Spears' ---
  Christina Aguilera (tag similarity: 0.87)
  Gwen Stefani (tag similarity: 0.87)
  Lady Gaga (tag similarity: 0.86)
  Madonna (tag similarity: 0.86)
  Hilary Duff (tag similarity: 0.85)

--- Tag-Based Playlist: 'Beirut' ---
  M. Ward (tag similarity: 0.85)
  Sufjan Stevens (tag similarity: 0.85)
  Bright Eyes (tag similarity: 0.83)
  Devendra Banhart (tag similarity: 0.83)
  Iron & Wine (tag similarity: 0.82)
